# Persona-Guard — Phase 0: Cloud model + activation smoke test

This notebook is the first real implementation step. It leaves the existing dashboard unchanged and runs a small, reproducible smoke test in Google Colab. It loads `Qwen/Qwen2.5-3B-Instruct`, generates a response to a harmless prompt, captures the final residual-stream vector at a configurable transformer layer, and saves run metadata plus the activation vector.

**What this does not do yet:** no secret-leakage experiment, no W1 calculation, no detector, no steering, and no claim of Persona-Guard effectiveness. Use only harmless synthetic prompts in this smoke test.

**Run:** Colab → Runtime → Change runtime type → GPU (if available), then run cells from top to bottom. If the GPU is unavailable, the notebook will stop with a clear message rather than trying to load a 3B model on a small CPU runtime.


In [ ]:
import sys, subprocess, json, os, platform, time
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "transformers>=4.45,<5", "accelerate", "safetensors"])
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, choose Runtime → Change runtime type → GPU, reconnect, and rerun.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
TARGET_LAYER = 18  # Must be checked against the loaded model architecture.
MAX_NEW_TOKENS = 48
PROMPT = "Explain in two sentences why reproducible experiments matter in machine learning. Do not include any passwords or private information."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, trust_remote_code=False
)
model.eval()
num_layers = len(model.model.layers)
if not (0 <= TARGET_LAYER < num_layers):
    raise ValueError(f"TARGET_LAYER={TARGET_LAYER} invalid; model has {num_layers} layers indexed 0..{num_layers-1}")
print({"model_id": MODEL_ID, "num_layers": num_layers, "target_layer": TARGET_LAYER, "dtype": str(next(model.parameters()).dtype)})


In [ ]:
captured = {}
def capture_hidden_states(module, args, output):
    # Decoder layers return a tensor or a tuple whose first item is the hidden states.
    h = output[0] if isinstance(output, tuple) else output
    captured["hidden"] = h.detach().float().cpu()

handle = model.model.layers[TARGET_LAYER].register_forward_hook(capture_hidden_states)
inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
# First capture the full prompt prefill (one activation vector per prompt token).
with torch.inference_mode():
    _ = model(**inputs, use_cache=False)
handle.remove()
hidden = captured.get("hidden")
if hidden is None:
    raise RuntimeError("Activation hook did not capture a tensor.")
# Then generate separately so the saved tensor is unambiguously prompt-only.
start = time.time()
with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, return_dict_in_generate=False)
elapsed = time.time() - start
generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
print("PROMPT:", PROMPT)
print("GENERATED TEXT:", generated_text)
print("Prompt activation tensor shape:", tuple(hidden.shape), "[batch, prompt_tokens, hidden_dim]")
print("Elapsed seconds:", round(elapsed, 2))


## Save a real, inspectable artifact
This records the model identifier, library versions, architecture/layer selection, prompt, generated response, token counts, runtime, and the captured activation tensor. The prompt is deliberately harmless. Review the output before sharing it; do not put real secrets or private information into prompts.

In [ ]:
from pathlib import Path
import numpy as np
from datetime import datetime, timezone

out_dir = Path("persona_guard_smoke_test")
out_dir.mkdir(exist_ok=True)
np.save(out_dir / "target_layer_hidden.npy", hidden.numpy())
metadata = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "transformers_version": __import__("transformers").__version__,
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "num_layers": num_layers,
    "target_layer_zero_based": TARGET_LAYER,
    "prompt": PROMPT,
    "generated_text": generated_text,
    "prompt_token_count": int(inputs["input_ids"].shape[1]),
    "generated_token_count": int(generated_ids.shape[0]),
    "max_new_tokens": MAX_NEW_TOKENS,
    "generation": {"do_sample": False},
    "elapsed_seconds": round(elapsed, 3),
    "activation_shape": list(hidden.shape),
    "activation_file": "target_layer_hidden.npy",
    "status": "smoke_test_only_not_a_leakage_evaluation"
}
(out_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", out_dir.resolve())
print("Files:", [p.name for p in out_dir.iterdir()])
print("Metadata status:", metadata["status"])


In [ ]:
# Download a ZIP containing the actual output artifacts from this run.
import shutil
archive = shutil.make_archive("persona_guard_smoke_test", "zip", root_dir=out_dir)
print("Artifact ZIP:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Not running inside Google Colab; download the ZIP from the file browser.")
